In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [2]:
pip install xgboost==2.0.0

Note: you may need to restart the kernel to use updated packages.


In [3]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [4]:
import numpy as np
import pandas as pd
import gc
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestRegressor
from scipy.special import logit, expit
from scipy.stats import rankdata
from xgboost import XGBClassifier

In [5]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/train.csv")
test  = pd.read_csv("/kaggle/input/competitions/playground-series-s6e3/test.csv")

In [6]:
train.shape
train.info()
train.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  object 
 16  PaperlessBilling  59

,id,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,594194.000000,594194.000000,594194.000000,594194.000000,594194.000000
mean,297096.500000,0.114102,36.577258,65.866223,2494.377057
std,171529.177262,0.317936,25.061922,31.067444,2353.916710
min,0.000000,0.000000,1.000000,18.250000,18.800000
25%,148548.250000,0.000000,12.000000,29.900000,639.650000
50%,297096.500000,0.000000,35.000000,74.100000,1433.650000
75%,445644.750000,0.000000,62.000000,90.800000,4263.800000
max,594193.000000,1.000000,72.000000,118.750000,8684.800000


In [7]:
TARGET = "Churn"
ID_COL = "id"

train[TARGET] = train[TARGET].map({"No": 0, "Yes": 1}).astype("int8")

In [8]:
train["TotalCharges"] = pd.to_numeric(train["TotalCharges"], errors="coerce")
test["TotalCharges"] = pd.to_numeric(test["TotalCharges"], errors="coerce")

train["TotalCharges"].fillna(train["TotalCharges"].median(), inplace=True)
test["TotalCharges"].fillna(test["TotalCharges"].median(), inplace=True)

In [9]:
y = train[TARGET]
test_ids = test[ID_COL]

X = train.drop(columns=[TARGET, ID_COL])
X_test = test.drop(columns=[ID_COL])

In [10]:
X["Tenure_Monthly"] = X["tenure"] * X["MonthlyCharges"]
X_test["Tenure_Monthly"] = X_test["tenure"] * X_test["MonthlyCharges"]

X["Charge_Ratio"] = X["TotalCharges"] / (X["tenure"] + 1)
X_test["Charge_Ratio"] = X_test["TotalCharges"] / (X_test["tenure"] + 1)

In [11]:
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
skf_enc = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for col in cat_cols:

    # Frequency encoding
    freq = X[col].value_counts(normalize=True)
    X[col] = X[col].map(freq)
    X_test[col] = X_test[col].map(freq).fillna(0)

X = X.astype("float32")
X_test = X_test.astype("float32")

gc.collect()

0

In [16]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(X), dtype="float32")
test_preds = np.zeros(len(X_test), dtype="float32")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    print(f"\n===== Fold {fold+1} =====")

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    xgb = XGBClassifier(
        n_estimators=50000,
        learning_rate=0.02,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=5,
        reg_alpha=1,
        min_child_weight=3,
        gamma=0.2,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="gpu_hist",
        predictor="gpu_predictor",
        device="cuda",
        random_state=fold * 99,
        early_stopping_rounds=1000
    )

    xgb.fit(
        X_train,
        y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )

    train_pred = xgb.predict_proba(X_train)[:,1]
    val_pred = xgb.predict_proba(X_val)[:,1]
    test_pred = xgb.predict_proba(X_test)[:,1]

    residual = y_train - train_pred

    rf = RandomForestRegressor(
        n_estimators=400,
        max_depth=8,
        min_samples_leaf=10,
        random_state=fold * 111,
        n_jobs=-1
    )

    rf.fit(X_train, residual)

    val_res = rf.predict(X_val)
    test_res = rf.predict(X_test)
 
    val_logit = logit(np.clip(val_pred, 1e-6, 1-1e-6))
    test_logit = logit(np.clip(test_pred, 1e-6, 1-1e-6))

    val_final = expit(val_logit + val_res)
    test_final = expit(test_logit + test_res)

    val_final = rankdata(val_final) / len(val_final)
    test_final = rankdata(test_final) / len(test_final)

    oof[val_idx] = val_final
    test_preds += test_final / skf.n_splits

    print("Fold AUC:", roc_auc_score(y_val, val_final))

print("\nFINAL CV AUC:", roc_auc_score(y, oof))


===== Fold 1 =====
Fold AUC: 0.9161010248609961

===== Fold 2 =====
Fold AUC: 0.917096249134681

===== Fold 3 =====
Fold AUC: 0.916553792071916

===== Fold 4 =====
Fold AUC: 0.9175420877928062

===== Fold 5 =====
Fold AUC: 0.9147922959966095

FINAL CV AUC: 0.9164171003725652


In [17]:
submission = pd.DataFrame({
    "id": test_ids,
    "Churn": test_preds
})

submission.to_csv("submission.csv", index=False)
print("Submission created successfully")

Submission created successfully
